# 실험 1 샘플링 — Quantinuum H2 (Nexus)

`sample_reference.ipynb`가 만든 5개 시점(`points.json`)의 회로를 Quantinuum H2 기기에서 각각 512 shots 샘플링한다. 학습은 하지 않는다. 흐름은 참고 레포 `Quantinuum_Nexus/30q_QAOA/qaoa_h2_hardware.ipynb`와 같다.

1. 설정 (기기 이름, shots, 제출 여부)
2. 5개 시점 읽기, pytket 회로 만들기, 로컬 검증 (요금 없음)
3. Nexus 로그인 (직접)
4. 회로 업로드 → syntax checker 컴파일(무료) → 기기용 컴파일 → 2큐비트 게이트 수 확인
5. 실행: `SUBMIT = True`일 때만. 시점마다 job 하나, job ID를 먼저 저장
6. 결과 받기: 저장된 job ID로 결과를 받아 counts 저장 (커널을 다시 켜도 이 셀부터 실행 가능)
7. 모든 backend 결과 모으기와 그림

결과는 `results_on_MI300X/extras/sampling_h2/grid5x5_inst0_seed120490/<기기 이름>/`에 저장되고, 그림은 같은 케이스 폴더에서 다른 backend 결과와 함께 다시 그려진다. 비트열은 q번째 문자 = 큐비트 q로 저장한다.

## 1. 설정

In [2]:
import datetime
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pytket.backends.backendresult import BackendResult
from pytket.circuit import Circuit
from pytket.utils.outcomearray import OutcomeArray

from common import save_json   # extras/sampling_h2/common.py

DEVICE_NAME = "H2-2"          # 실행할 기기 이름 (예: H2-1, H2-2). 결과 폴더 이름이 된다
SYNTAX_CHECKER = "H2-2SC"     # 같은 계열의 syntax checker (무료 컴파일 점검)
PROJECT_NAME = "SAFE_TTA_exp1_sampling"
SHOTS = 512
OPTIMISATION_LEVEL = 2
SUBMIT_POINTS = ["p1_initial", "p2_exact_only", "p3_safe_pretrained", "p4_safe_distilled", "p5_safe_finetuned"]

CASE = "grid5x5_inst0_seed120490"
REPO = Path.cwd().parents[1]      # 이 노트북은 extras/sampling_h2/ 폴더에서 실행한다
OUT = REPO / "results_on_MI300X" / "extras" / "sampling_h2" / CASE
OUT_DEV = OUT / DEVICE_NAME

## 2. 5개 시점과 pytket 회로

`points.json`(sample_reference.ipynb가 만듦)의 게이트 목록을 pytket 게이트로 옮긴다. pytket 각도는 π 단위(반바퀴)이므로 θ/π를 넣는다.

| circuit.json | 의미 | pytket |
|---|---|---|
| H | Hadamard | `H(q)` |
| RZZ(θ) | exp(−iθ Z⊗Z/2) | `ZZPhase(θ/π, u, v)` (H2의 기본 2큐비트 게이트) |
| RZ(θ) | exp(−iθ Z/2) | `Rz(θ/π, q)` |
| RX(θ) | exp(−iθ X/2) | `Rx(θ/π, q)` |

측정은 큐비트 q → 고전 비트 c[q]. H2는 모든 큐비트가 서로 연결돼 있어 배치나 SWAP을 신경 쓰지 않아도 된다.

In [3]:
points_doc = json.loads((OUT / "points.json").read_text())
points = points_doc["points"]
prob = json.loads((REPO / "problems" / f"{CASE}.json").read_text())
n = prob["n_qubits"]

circuits = {}
for key, p in points.items():
    theta = p["theta"]
    tk = Circuit(n, n, name=f"exp1-{key}")
    for g in p["gates"]:
        if g["gate"] == "H":
            tk.H(g["wires"][0])
        elif g["gate"] == "RZZ":
            tk.ZZPhase(theta[g["param"]] / np.pi, *g["wires"])
        elif g["gate"] == "RZ":
            tk.Rz(theta[g["param"]] / np.pi, g["wires"][0])
        elif g["gate"] == "RX":
            tk.Rx(theta[g["param"]] / np.pi, g["wires"][0])
    for q in range(n):
        tk.Measure(q, q)
    circuits[key] = tk
    print(f"{key:20s} qubits={tk.n_qubits}  2-qubit gates={tk.n_2qb_gates()}  depth={tk.depth()}  exact AR={p['ar_exact']:.4f}")

p1_initial           qubits=25  2-qubit gates=216  depth=44  exact AR=0.5081
p2_exact_only        qubits=25  2-qubit gates=216  depth=44  exact AR=0.8101
p3_safe_pretrained   qubits=25  2-qubit gates=216  depth=44  exact AR=0.8416
p4_safe_distilled    qubits=25  2-qubit gates=72  depth=24  exact AR=0.9475
p5_safe_finetuned    qubits=25  2-qubit gates=72  depth=24  exact AR=0.9663


### 로컬 검증 (요금 없음)

pytket의 자체 시뮬레이터는 25큐비트를 지원하지 않는다. 그래서 같은 변환 규칙을 3x3 회로(실험 3의 시작점 두 개)에 적용해, pytket statevector로 계산한 에너지가 PennyLane으로 계산해 둔 값과 같은지 본다. 이어서 pytket 결과의 비트 순서(c[q]가 q번째 자리)를 확인한다.

In [4]:
sp = REPO / "results_on_MI300X" / "exp3_quantum_3x3" / "grid3x3_inst0_seed120490" / "start_points"
c3 = json.loads((sp / "circuit.json").read_text())
s3 = json.loads((sp / "params.json").read_text())
for gates, theta, ref in [(c3["gates"], s3["theta_init"], s3["exact_energy"]["exact_only_start"]),
                          (c3["pruned_gates"], s3["theta_safe_start"], s3["exact_energy"]["safe_start"])]:
    tk = Circuit(9)
    for g in gates:
        if g["gate"] == "H":
            tk.H(g["wires"][0])
        elif g["gate"] == "RZZ":
            tk.ZZPhase(theta[g["param"]] / np.pi, *g["wires"])
        elif g["gate"] == "RZ":
            tk.Rz(theta[g["param"]] / np.pi, g["wires"][0])
        elif g["gate"] == "RX":
            tk.Rx(theta[g["param"]] / np.pi, g["wires"][0])
    prob_amp = np.abs(tk.get_statevector()) ** 2             # pytket: 큐비트 0이 가장 높은 자리
    e = 0.0
    for idx in np.flatnonzero(prob_amp > 0):
        s = 1 - 2 * np.array([(idx >> (8 - q)) & 1 for q in range(9)])
        e += prob_amp[idx] * (sum(w * s[u] * s[v] for u, v, w in c3["edges"]) + sum(h * s[q] for q, h in c3["fields"]))
    assert abs(e - ref) < 1e-9, (e, ref)
    print(f"3x3 check: pytket <H> = {e:+.10f}   PennyLane {ref:+.10f}")

# 비트 순서: 큐비트 0만 1인 측정 결과 -> counts 키 (1, 0, 0) -> 문자열 '100'
check = Circuit(3, 3)
res = BackendResult(shots=OutcomeArray.from_readouts([[1, 0, 0]]), c_bits=check.bits)
key = list(res.get_counts())[0]
assert "".join(str(int(b)) for b in key) == "100", key
print("pytket counts key", key, "-> stored as '100'")

3x3 check: pytket <H> = -0.1747624990   PennyLane -0.1747624990
3x3 check: pytket <H> = -7.4149925623   PennyLane -7.4149925623
pytket counts key (1, 0, 0) -> stored as '100'


## 3. Nexus 로그인 (직접 실행)

로그인은 직접 한다. 브라우저 로그인이 안 되는 환경이면 qnexus의 `qnx.login_with_credentials()`도 있다.

In [5]:
import qnexus as qnx

qnx.login()
project = qnx.projects.get_or_create(name=PROJECT_NAME)
qnx.context.set_active_project(project)
config = qnx.QuantinuumConfig(device_name=DEVICE_NAME, target_2qb_gate="ZZPhase")
sc_config = qnx.QuantinuumConfig(device_name=SYNTAX_CHECKER, target_2qb_gate="ZZPhase")
print("project:", PROJECT_NAME, "| device:", DEVICE_NAME, "| syntax checker:", SYNTAX_CHECKER)

Already logged in. Tokens are valid.
project: SAFE_TTA_exp1_sampling | device: H2-2 | syntax checker: H2-2SC


## 4. 업로드와 컴파일 (실행 전 점검)

회로를 올리고, 무료 syntax checker로 먼저 컴파일한 뒤 실제 기기용으로 컴파일한다. 컴파일된 회로의 2큐비트 게이트 수를 보고, 비용(HQC)은 Nexus 화면에서 확인한 다음 제출 여부를 정한다.

In [6]:
prefix = f"exp1-sampling-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
labels = list(circuits)
ref_circuits = {k: qnx.circuits.upload(circuit=circuits[k], name=f"{prefix}-{k}") for k in labels}

sc_job = qnx.start_compile_job(programs=[ref_circuits[k] for k in labels], backend_config=sc_config,
                               optimisation_level=OPTIMISATION_LEVEL, name=f"sc-{prefix}")
qnx.jobs.wait_for(sc_job)
print("syntax checker compile OK on", SYNTAX_CHECKER)

compile_job = qnx.start_compile_job(programs=[ref_circuits[k] for k in labels], backend_config=config,
                                    optimisation_level=OPTIMISATION_LEVEL, name=f"compile-{prefix}")
qnx.jobs.wait_for(compile_job)
compiled_results = qnx.jobs.results(compile_job)
ref_compiled = {labels[i]: compiled_results[i].get_output() for i in range(len(labels))}
for k in labels:
    c = ref_compiled[k].download_circuit()
    print(f"{k:20s} compiled: 2-qubit gates={c.n_2qb_gates()}  depth={c.depth()}")
print(f"{SHOTS} shots per point -- check the HQC cost in the Nexus UI before setting SUBMIT = True")

syntax checker compile OK on H2-2SC
p1_initial           compiled: 2-qubit gates=216  depth=41
p2_exact_only        compiled: 2-qubit gates=216  depth=60
p3_safe_pretrained   compiled: 2-qubit gates=214  depth=61
p4_safe_distilled    compiled: 2-qubit gates=71  depth=27
p5_safe_finetuned    compiled: 2-qubit gates=71  depth=27
512 shots per point -- check the HQC cost in the Nexus UI before setting SUBMIT = True


## 5. 실행 (`SUBMIT = True`일 때만)

아직 샘플도 없고 제출하지도 않은 시점을 모두 모아 **job 하나**로 보낸다. `start_execute_job`의 `programs`에 컴파일된 회로 목록을, `n_shots`에 회로별 shots 목록을 넣으면 한 job 안에서 회로마다 따로 실행되고 결과도 회로별로 나온다 (Qiskit에서 PUB 여러 개를 한 job에 넣는 것과 같은 방식). job ID와 각 회로의 ID를 바로 `jobs.json`에 저장하므로, 결과를 기다리다 끊겨도 6번 셀에서 다시 받아 올 수 있다.

In [7]:
OUT_DEV.mkdir(parents=True, exist_ok=True)
jobs_path, counts_path = OUT_DEV / "jobs.json", OUT_DEV / "counts.json"
jobs = json.loads(jobs_path.read_text()) if jobs_path.exists() else {"device": DEVICE_NAME, "shots": SHOTS, "jobs": []}
done = json.loads(counts_path.read_text())["counts"] if counts_path.exists() else {}

SUBMIT = True                # True로 바꿔야 실제 기기에 제출한다 (크레딧 사용)
# 아직 샘플도 없고 제출도 안 한 시점만 골라 job 하나에 담는다
submitted = {k for j in jobs["jobs"] for k in j["points"]}
pending = [k for k in SUBMIT_POINTS if k not in done and k not in submitted]
if SUBMIT and pending:
    name = f"exec-{prefix}"
    job = qnx.start_execute_job(programs=[ref_compiled[k] for k in pending],   # 회로 여러 개 = job 하나
                                n_shots=[SHOTS] * len(pending),                # 회로마다 SHOTS번
                                backend_config=config, name=name)
    jobs["jobs"].append({"job_id": str(job.id), "name": name, "points": pending,
                         "program_ids": [str(ref_compiled[k].id) for k in pending]})   # 결과를 시점에 맞출 때 쓴다
    save_json(jobs_path, jobs)
    print(f"submitted 1 job with {len(pending)} circuits {pending} -> {qnx.jobs.status(job).status}")
elif SUBMIT:
    print("nothing left to submit (every point is already submitted or sampled)")
else:
    print("SUBMIT is False: nothing submitted")

submitted 1 job with 5 circuits ['p1_initial', 'p2_exact_only', 'p3_safe_pretrained', 'p4_safe_distilled', 'p5_safe_finetuned'] -> JobStatusEnum.SUBMITTED


## 6. 결과 받기

`jobs.json`의 job ID로 결과를 받는다. 한 job의 결과는 회로별로 나오며, 각 결과의 입력 회로 ID(`get_input().id`)로 어느 시점의 결과인지 맞춘다. pytket counts의 키 (c[0], c[1], ...)를 이어 붙여 q번째 문자 = 큐비트 q 문자열로 저장한다. 커널을 다시 켠 뒤에는 1번 설정 셀, 3번 로그인 셀, 그리고 이 셀만 실행하면 된다.

In [ ]:
jobs_path, counts_path = OUT_DEV / "jobs.json", OUT_DEV / "counts.json"
jobs = json.loads(jobs_path.read_text())
data = json.loads(counts_path.read_text()) if counts_path.exists() else {
    "backend": DEVICE_NAME, "shots": SHOTS, "counts": {}, "jobs": {}}
for info in jobs["jobs"]:
    if all(k in data["counts"] for k in info["points"]):
        continue
    job = qnx.jobs.get(id=info["job_id"])
    qnx.jobs.wait_for(job)
    point_of = dict(zip(info["program_ids"], info["points"]))      # 입력 회로 ID -> 시점
    for r in qnx.jobs.results(job):                                 # 회로별 결과
        k = point_of[str(r.get_input().id)]
        counts = r.download_result().get_counts()
        data["counts"][k] = {"".join(str(int(b)) for b in key): int(c) for key, c in counts.items()}
        data["jobs"][k] = {"job_id": info["job_id"], "name": info["name"]}
        print(f"{k:20s} {sum(data['counts'][k].values())} shots downloaded")
    save_json(counts_path, data)

## 모든 backend 결과 모으기와 그림

`OUT` 아래에서 `counts.json`이 있는 하위 폴더(= backend)를 모두 읽는다. 지금까지 모인 backend만 그림에 들어가므로, 어느 노트북을 나중에 돌려도 그림과 `summary.json`이 다시 만들어진다. backend별 원본 counts는 건드리지 않는다.

- 샘플 하나의 에너지: 비트열에서 s_q = 1 − 2·bit_q, E = Σ J_ij s_i s_j + Σ h_i s_i
- 샘플 AR = (E_max − 샘플 평균 에너지) / (E_max − E_min), brute force E_min, E_max는 `problems/<케이스>.json`
- 바닥 상태 비율 = brute force 바닥 상태 비트열이 나온 횟수 / shots
- `sampling_hist.png`: backend별 한 줄, 왼쪽은 exact-only 경로(①②), 오른쪽은 SAFE 경로(①③④⑤)의 샘플 에너지 분포. 파란 점선은 E_min
- `sampling_ar.png`: 시점별 AR 막대. 회색은 실험 1의 노이즈 없는 기댓값, 나머지는 backend별 샘플 AR

In [ ]:
points_doc = json.loads((OUT / "points.json").read_text())
points = points_doc["points"]
prob = json.loads((REPO / "problems" / f"{CASE}.json").read_text())
n, e_min, e_max = prob["n_qubits"], prob["e_min"], prob["e_max"]
ground = "".join(str(b) for b in prob["ground_state_bits"])          # 문자 q = 큐비트 q
U = np.array([u for u, _, _ in prob["edges"]]); V = np.array([v for _, v, _ in prob["edges"]])
W = np.array([w for _, _, w in prob["edges"]])
Q = np.array([q for q, _ in prob["fields"]]); HF = np.array([h for _, h in prob["fields"]])

# backend 폴더 순서: 노이즈 없는 기준, fake, 그 밖(H2 등)
found = sorted(d.name for d in OUT.iterdir() if (d / "counts.json").exists())
backends = [b for b in ["noiseless", "fake_nighthawk"] if b in found] + [b for b in found if b not in ("noiseless", "fake_nighthawk")]
colors = {"noiseless": "tab:blue", "fake_nighthawk": "tab:green"}
extra = iter(["tab:red", "tab:purple", "tab:brown"])
for b in backends:
    colors.setdefault(b, next(extra))

# 샘플별 에너지와 요약
energies, summary = {}, {"case": CASE, "c": points_doc["c"], "e_min": e_min, "e_max": e_max, "backends": {}}
for b in backends:
    data = json.loads((OUT / b / "counts.json").read_text())
    energies[b], summary["backends"][b] = {}, {}
    for key, counts in data["counts"].items():
        bits = np.array([[int(ch) for ch in s] for s in counts])            # (서로 다른 비트열 수, n)
        mult = np.array(list(counts.values()))                              # 각 비트열이 나온 횟수
        s = 1 - 2 * bits
        e = (s[:, U] * s[:, V]) @ W + s[:, Q] @ HF                           # 비트열별 에너지
        energies[b][key] = (e, mult)
        shots = int(mult.sum())
        mean = float((e * mult).sum() / shots)
        summary["backends"][b][key] = {
            "shots": shots, "mean_energy": mean,
            "std_energy": float(np.sqrt((((e - mean) ** 2) * mult).sum() / shots)),
            "ar_sampled": (e_max - mean) / (e_max - e_min),
            "best_energy": float(e.min()), "best_ar": (e_max - float(e.min())) / (e_max - e_min),
            "p_ground_state": counts.get(ground, 0) / shots,
            "ar_exact_expectation": points[key]["ar_exact"]}
save_json(OUT / "summary.json", summary)
for b in backends:
    for key in points:
        if key in summary["backends"][b]:
            r = summary["backends"][b][key]
            print(f"{b:16s} {key:20s} AR sampled={r['ar_sampled']:.4f} (exact {r['ar_exact_expectation']:.4f})  "
                  f"best E={r['best_energy']:+.3f}  P(ground)={r['p_ground_state']:.4f}")

# 그림 1: 샘플 에너지 분포
paths = [("exact-only path", ["p1_initial", "p2_exact_only"]),
         ("SAFE path", ["p1_initial", "p3_safe_pretrained", "p4_safe_distilled", "p5_safe_finetuned"])]
point_colors = {"p1_initial": "gray", "p2_exact_only": "black", "p3_safe_pretrained": "gold",
                "p4_safe_distilled": "darkorange", "p5_safe_finetuned": "red"}
bins = np.linspace(e_min - 1, e_max + 1, 70)
fig, axes = plt.subplots(len(backends), 2, figsize=(12, 3.0 * len(backends)), squeeze=False, sharex=True)
for row, b in enumerate(backends):
    for col, (title, keys) in enumerate(paths):
        ax = axes[row][col]
        for key in keys:
            if key in energies[b]:
                e, mult = energies[b][key]
                ax.hist(e, bins=bins, weights=mult, histtype="stepfilled", alpha=0.35, color=point_colors[key],
                        label=points[key]["label"].replace("\n", " "))
                ax.hist(e, bins=bins, weights=mult, histtype="step", color=point_colors[key])
        ax.axvline(e_min, color="tab:blue", linestyle="--", linewidth=1, label="brute-force E_min")
        ax.set_title(f"{b}: {title}", fontsize=9)
        ax.set_ylabel("shot count")
        ax.legend(fontsize=7)
for ax in axes[-1]:
    ax.set_xlabel("Ising energy of each shot")
fig.suptitle(f"{CASE}, c = {points_doc['c']}, {points_doc['shots']} shots per point", fontsize=10)
fig.tight_layout()
fig.savefig(OUT / "sampling_hist.png", dpi=150)

# 그림 2: 시점별 AR 막대 (회색 = 실험 1 노이즈 없는 기댓값)
keys = list(points)
series = [("exact expectation (exp1)", [points[k]["ar_exact"] for k in keys], "lightgray")]
series += [(f"{b} sampled", [summary["backends"][b].get(k, {}).get("ar_sampled", np.nan) for k in keys], colors[b]) for b in backends]
x = np.arange(len(keys))
width = 0.8 / len(series)
fig, ax = plt.subplots(figsize=(11, 4.6))
for i, (name, vals, color) in enumerate(series):
    xs = x - 0.4 + width * (i + 0.5)
    ax.bar(xs, vals, width, color=color, label=name)
    for xv, val in zip(xs, vals):
        if not np.isnan(val):
            ax.text(xv, val + 0.01, f"{val:.3f}", ha="center", va="bottom", fontsize=7, rotation=90,
                    bbox=dict(facecolor="white", edgecolor="none", pad=0.5), zorder=3)
ax.axhline(1.0, color="tab:blue", linestyle="--", linewidth=1, label="brute-force optimum (AR = 1)")
ax.set_xticks(x)
ax.set_xticklabels([points[k]["label"] for k in keys], fontsize=8)
ax.set_ylim(0, 1.3)
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel("approximation ratio")
ax.set_title(f"{CASE}, c = {points_doc['c']}, {points_doc['shots']} shots per point", fontsize=9)
ax.legend(fontsize=7, loc="upper left", ncol=len(series) + 1)
fig.tight_layout()
fig.savefig(OUT / "sampling_ar.png", dpi=150)
print("saved", (OUT / "sampling_hist.png").relative_to(REPO), "and", (OUT / "sampling_ar.png").relative_to(REPO))